In [1]:
import pandas as pd
from pathlib import Path

In [2]:
#Entrando no caminho dos arquivos
DATA_PATH = Path("../data/raw")

files = list(DATA_PATH.glob("*.csv"))

for file in files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [3]:
# Carregando os datasets em um dicionário
# CEP é lido como string: como inteiro os zeros à esquerda se perdem (ex.: 01046 -> 1046)
zip_dtypes = {
    "customer_zip_code_prefix": str,
    "seller_zip_code_prefix": str,
    "geolocation_zip_code_prefix": str,
}

datasets = {}

for file in files:
    name = file.stem
    datasets[name] = pd.read_csv(file, dtype=zip_dtypes)

print(f"{len(datasets)} datasets carregados.")

9 datasets carregados.


In [4]:
# Printando a forma de cada dataset
for name, df in datasets.items():
    print(f"{name}: {df.shape}")

olist_customers_dataset: (99441, 5)
olist_geolocation_dataset: (1000163, 5)
olist_orders_dataset: (99441, 8)
olist_order_items_dataset: (112650, 7)
olist_order_payments_dataset: (103886, 5)
olist_order_reviews_dataset: (99224, 7)
olist_products_dataset: (32951, 9)
olist_sellers_dataset: (3095, 4)
product_category_name_translation: (71, 2)


In [5]:
# Carregando dataframes
reviews_df = pd.DataFrame(datasets["olist_order_reviews_dataset"])
orders_df = pd.DataFrame(datasets["olist_orders_dataset"])
reviews_df.shape

(99224, 7)

In [6]:
reviews_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [7]:
#Avaliando se existe valores nulos
reviews_df.isna().sum()

review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [8]:
# Percentual de nulos nas colunas de comentário (cliente pode dar só a nota)
reviews_df[['review_comment_title', 'review_comment_message']].isna().mean().mul(100).round(2)

review_comment_title      88.34
review_comment_message    58.70
dtype: float64

In [9]:
#Avaliando se existe dados duplicados
print(f"Linhas totalmente duplicadas: {reviews_df.duplicated().sum()}")
print(f"review_id duplicados: {reviews_df['review_id'].duplicated().sum()}")
print(f"Duplicidade em review_id + order_id: {reviews_df.duplicated(subset=['review_id', 'order_id']).sum()}")

Linhas totalmente duplicadas: 0
review_id duplicados: 814


Duplicidade em review_id + order_id: 0

In [10]:
# review_id repetido: o mesmo review_id aparece em pedidos diferentes?
dup_ids = reviews_df[reviews_df['review_id'].duplicated(keep=False)]

print(f"Linhas envolvidas: {len(dup_ids)}")
print(f"review_id distintos envolvidos: {dup_ids['review_id'].nunique()}")
print(f"review_id com mais de um order_id: {(dup_ids.groupby('review_id')['order_id'].nunique() > 1).sum()}")
dup_ids.sort_values('review_id').head(6)

Linhas envolvidas: 1603
review_id distintos envolvidos: 789
review_id com mais de um order_id: 789


,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53


In [11]:
# Pedidos com mais de uma avaliação
reviews_per_order = reviews_df.groupby('order_id').size()

print(f"Pedidos com mais de uma avaliação: {(reviews_per_order > 1).sum()}")
print(f"Máximo de avaliações em um pedido: {reviews_per_order.max()}")

Pedidos com mais de uma avaliação: 547


Máximo de avaliações em um pedido: 3


In [12]:
# review_score: valores de 1 a 5
reviews_df['review_score'].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [13]:
# Comentários: strings vazias e quebras de linha
for col in ['review_comment_title', 'review_comment_message']:
    text = reviews_df[col].dropna()
    print(f"{col}: vazios = {(text.str.strip() == '').sum()} | com quebra de linha = {text.str.contains(r'[\r\n]').sum()}")

review_comment_title: vazios = 2 | com quebra de linha = 0
review_comment_message: vazios = 27 | com quebra de linha = 3852


In [14]:
# Datas estão como string
cols_data = ['review_creation_date', 'review_answer_timestamp']

reviews_df[cols_data].dtypes

review_creation_date       str
review_answer_timestamp    str
dtype: object

In [15]:
# Integridade com orders
print(f"Avaliações cujo order_id não existe em orders: {(~reviews_df['order_id'].isin(orders_df['order_id'])).sum()}")
print(f"Pedidos sem avaliação: {(~orders_df['order_id'].isin(reviews_df['order_id'])).sum()}")

Avaliações cujo order_id não existe em orders: 0
Pedidos sem avaliação: 768


In [16]:
#Fazendo copia do dataframe para não alterar o original
reviews_clean = reviews_df.copy()

In [17]:
# Convertendo as colunas de data de string para datetime
for col in cols_data:
    reviews_clean[col] = pd.to_datetime(reviews_clean[col])

reviews_clean[cols_data].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [18]:
# Comentários: remove quebras de linha/espaços repetidos e transforma texto vazio em nulo
for col in ['review_comment_title', 'review_comment_message']:
    reviews_clean[col] = reviews_clean[col].str.replace(r'\s+', ' ', regex=True).str.strip()
    reviews_clean.loc[reviews_clean[col] == '', col] = pd.NA

In [19]:
# Comparando nulos antes e depois
pd.DataFrame({
    "nulos_antes": reviews_df[['review_comment_title', 'review_comment_message']].isna().sum(),
    "nulos_depois": reviews_clean[['review_comment_title', 'review_comment_message']].isna().sum(),
})

,nulos_antes,nulos_depois
review_comment_title,87656,87658
review_comment_message,58247,58274


In [20]:
# Validações pós-tratamento
print(f"Linhas: {reviews_clean.shape[0]} (original: {reviews_df.shape[0]})")
print(f"review_score fora de 1-5: {(~reviews_clean['review_score'].between(1, 5)).sum()}")
print(f"Resposta anterior à criação: {(reviews_clean['review_answer_timestamp'] < reviews_clean['review_creation_date']).sum()}")
print(f"Duplicidade em review_id + order_id: {reviews_clean.duplicated(subset=['review_id', 'order_id']).sum()}")
print(f"Comentários com quebra de linha: {reviews_clean['review_comment_message'].dropna().str.contains(r'[\r\n]').sum()}")
print(f"Avaliações sem correspondência em orders: {(~reviews_clean['order_id'].isin(orders_df['order_id'])).sum()}")
print(f"Data mínima / máxima de criação: {reviews_clean['review_creation_date'].min()} / {reviews_clean['review_creation_date'].max()}")

assert reviews_clean.shape == reviews_df.shape
assert reviews_clean['review_score'].equals(reviews_df['review_score'])
assert reviews_clean['review_creation_date'].dtype.kind == 'M'
assert not reviews_clean.duplicated(subset=['review_id', 'order_id']).any()

Linhas: 99224 (original: 99224)
review_score fora de 1-5: 0
Resposta anterior à criação: 0
Duplicidade em review_id + order_id: 0
Comentários com quebra de linha: 0
Avaliações sem correspondência em orders: 0
Data mínima / máxima de criação: 2016-10-02 00:00:00 / 2018-08-31 00:00:00


In [21]:
# Salvando dataframe tratado em pasta processed
reviews_clean.to_csv('../data/processed/olist_order_reviews_dataset_clean.csv', index=False)

## 📌 Conclusão da etapa — Order Reviews

### 🔎 O que foi encontrado?

A tabela `order_reviews` possui **99.224 registros e 7 colunas**.

Durante o profiling, foram identificados:

- `review_comment_title` com **88,34%** de nulos e `review_comment_message` com **58,70%**: é esperado, pois muitos clientes só dão a nota;
- `review_id` **não é único**: 814 linhas repetidas (789 `review_id` distintos), sempre ligadas a pedidos diferentes. A combinação `review_id + order_id` é única;
- **547 pedidos** com mais de uma avaliação (máximo de 3);
- `review_score` somente entre 1 e 5 e resposta nunca anterior à criação;
- 2 títulos e 27 mensagens com texto vazio e **3.852 mensagens com quebra de linha**;
- Colunas de data armazenadas como `string`;
- **768 pedidos** sem avaliação, e todas as avaliações possuem pedido em `orders`.

### 🛠️ O que foi feito?

- `review_creation_date` e `review_answer_timestamp` convertidas para `datetime`;
- Nos comentários, quebras de linha e espaços repetidos viraram um espaço, e textos vazios viraram nulo (+2 nulos no título e +27 na mensagem). Isso evita problemas de importação do CSV no banco e no Power BI;
- Os `review_id` repetidos foram **mantidos**, pois representam avaliações de pedidos diferentes.

Nenhum registro foi removido.

### ✅ Como foi validado?

- A quantidade de registros permaneceu em **99.224** e `review_score` ficou idêntico ao original;
- `review_id + order_id` sem duplicidade;
- Nenhum comentário com quebra de linha;
- Datas de criação de **2016-10-02 a 2018-08-31**.

### 📁 Resultado

O dataset tratado foi salvo em:

`data/processed/olist_order_reviews_dataset_clean.csv`

> ⚠️ Para a nota média por pedido, agregar por `order_id` antes de calcular, já que 547 pedidos têm mais de uma avaliação.